<h1>RAZ Systems </h1>

# Assignment 2 -- Multi-Agent Email Pipeline: Skyline Realty

Same building blocks as the lesson -- streaming, parallel agents, an evaluator, tools, an orchestrator, and a handoff -- just a different business. Instead of HNW client outreach for Apex Capital, you'll build a **real-estate listing outreach pipeline** for **Skyline Realty**.

### Setup

You should already have SendGrid configured from the lesson. If not: visit https://sendgrid.com/, create a free account, generate an API key, add `SENDGRID_API_KEY=xxxx` to your `.env`, and verify a sender email under Settings >> Sender Authentication.

In [ ]:
# --- Imports ---
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

load_dotenv(override=True)

## Part 1a -- Single Agent, Streamed Output

**TODO:** create three listing-copywriter personas for Skyline Realty -- e.g. a **luxury-focused** writer, a **family-focused** writer, and an **investor-focused** writer. Same goal, different tone, so we can compare approaches later.

**Concept reminder:** `Runner.run_streamed()` lets you see the response being built token-by-token.

In [ ]:
# --- TODO: Three listing-copywriter personas for Skyline Realty ---

instructions1 = (
    ___  # TODO: a LUXURY-focused real-estate copywriter persona for Skyline Realty
)

instructions2 = (
    ___  # TODO: a FAMILY-focused real-estate copywriter persona for Skyline Realty
)

instructions3 = (
    ___  # TODO: an INVESTOR-focused real-estate copywriter persona for Skyline Realty
)

In [ ]:
# --- TODO: Create the three listing agents ---

listing_agent1 = Agent(
    name=___,            # TODO
    instructions=instructions1,
    model="gpt-4o-mini",
)

listing_agent2 = Agent(
    name=___,            # TODO
    instructions=instructions2,
    model="gpt-4o-mini",
)

listing_agent3 = Agent(
    name=___,            # TODO
    instructions=instructions3,
    model="gpt-4o-mini",
)

In [ ]:
# --- TODO: Stream a single draft to see what one agent produces ---
# Runner.run_streamed() prints tokens as they arrive.

async def run():
    result = Runner.run_streamed(___, input=___)   # TODO: pick one listing agent + a listing prompt
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)

task = asyncio.ensure_future(run())
await task

## Part 1b -- Three Agents in Parallel

**Upgrade:** instead of running one listing agent at a time, fire all three simultaneously with `asyncio.gather()`.

**TODO:** fill in the prompt and the three `Runner.run(...)` calls.

In [ ]:
# --- TODO: Run all three listing agents in parallel and compare drafts ---

message = ___  # TODO: e.g. "Write a listing description for a 3-bed house at 12 Oak Lane"

with trace("Parallel listing drafts"):
    results = await asyncio.gather(
        Runner.run(___, message),  # TODO: listing_agent1
        Runner.run(___, message),  # TODO: listing_agent2
        Runner.run(___, message),  # TODO: listing_agent3
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

## Part 1c -- Add a Picker Agent (Agent as Evaluator)

**Upgrade:** a fourth agent (`buyer_picker`) receives all three drafts and selects the strongest one, from the perspective of a real prospective buyer.

**TODO:** write the evaluator's instructions.

In [ ]:
# --- TODO: A picker agent selects the strongest listing draft ---

buyer_picker = Agent(
    name="Prospective Buyer Evaluator",
    instructions=(
        ___  # TODO: instructions for evaluating from the perspective of a serious home buyer.
        # Should pick the single description most likely to prompt a viewing request,
        # and reply with the selected description only (no explanation).
    ),
    model="gpt-4o-mini",
)

In [ ]:
# --- TODO: Generate, evaluate, and print the best draft ---

message = ___  # TODO

with trace("Listing draft selection"):
    results = await asyncio.gather(
        Runner.run(listing_agent1, message),
        Runner.run(listing_agent2, message),
        Runner.run(listing_agent3, message),
    )
    outputs = [result.final_output for result in results]

    drafts = "Listing drafts:\n\n" + "\n\nDraft:\n\n".join(outputs)

    best = await Runner.run(___, drafts)  # TODO: pass your picker agent

    print(f"Best listing description:\n{best.final_output}")

## Part 2 -- Tools: `@function_tool` and `.as_tool()`

**Upgrade:** instead of just printing the winning draft, agents can now *act* -- by calling tools.

**TODO:**
- Write a `send_listing_email` function tool
- Convert each listing agent into a tool with `.as_tool()`

In [ ]:
# --- TODO: a function tool that sends the listing as a plain-text email ---

@function_tool
def send_listing_email(body: str) -> Dict[str, str]:
    """ ___ """  # TODO: docstring describing what this tool does
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(___)  # TODO: your verified sender
    to_email = To(___)       # TODO: your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "New Listing - Skyline Realty", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
# --- TODO: convert each listing agent into a tool, and gather everything ---

description = ___  # TODO: e.g. "Write a real-estate listing description"

tool1 = listing_agent1.as_tool(tool_name=___, tool_description=description)  # TODO: tool_name
tool2 = listing_agent2.as_tool(tool_name=___, tool_description=description)  # TODO: tool_name
tool3 = listing_agent3.as_tool(tool_name=___, tool_description=description)  # TODO: tool_name

tools = [tool1, tool2, tool3, send_listing_email]
tools

## Part 3a -- Orchestrator Agent (Tools Only)

**Upgrade:** a **Listings Manager** orchestrator agent is given all three listing-agent tools plus `send_listing_email`. It independently decides to call all three, evaluate the drafts, pick the winner, and send -- no explicit sequential logic in our code.

**TODO:** write the orchestrator's instructions, following the same careful step-by-step pattern from the lesson (generate all three drafts, evaluate, send exactly ONE email, never duplicate).

In [ ]:
# --- TODO: Listings Manager orchestrator ---

listings_manager_instructions = """
___
"""  # TODO: Head-of-department style instructions:
# 1. Generate Drafts: use all three listing agent tools, don't proceed until all three are ready.
# 2. Evaluate and Select: choose the strongest draft.
# 3. Send: use send_listing_email to send ONLY the winning draft.
# Crucial rules: must use the tools to draft (don't write it yourself); send exactly ONE email.

listings_manager = Agent(
    name="Listings Manager",
    instructions=listings_manager_instructions,
    tools=tools,
    model="gpt-4o-mini",
)

message = ___  # TODO: e.g. "Send a listing email for 12 Oak Lane, a 3-bed family home"

with trace("Skyline Realty listing - Listings Manager"):
    result = await Runner.run(listings_manager, message)

### Check the trace

https://platform.openai.com/traces

## Part 3b -- Handoffs + Listing Operations Manager

**Final upgrade:** instead of sending plain text directly, the Listings Manager now **hands off** the winning draft to a dedicated `Listing Operations Manager`. That agent uses two further sub-agents -- a Subject Line Writer and an HTML Formatter -- before sending the polished email.

**Reminder -- tools vs handoffs:**
- **Tools** -- control *returns* to the calling agent after the tool finishes
- **Handoffs** -- control passes *across* to the receiving agent permanently

**TODO:** build the sub-agents, the operations manager, and rewire the Listings Manager to hand off instead of sending directly.

In [ ]:
# --- TODO: sub-agents for formatting ---

subject_instructions = (
    ___  # TODO: writes compelling, non-salesy subject lines for a listing email
)

html_instructions = (
    ___  # TODO: converts a plain-text listing email body into a clean, simple HTML layout
)

subject_writer = Agent(name="Subject Line Writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject line for a listing email")

html_converter = Agent(name="HTML Email Formatter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a plain-text email body to HTML")

In [ ]:
# --- TODO: the actual send-as-HTML function tool ---

@function_tool
def send_html_listing_email(subject: str, html_body: str) -> Dict[str, str]:
    """ ___ """  # TODO: docstring
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(___)  # TODO
    to_email = To(___)       # TODO
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [ ]:
# --- TODO: Listing Operations Manager ---

ops_tools = [subject_tool, html_tool, send_html_listing_email]

ops_instructions = (
    ___  # TODO: an Email Operations Manager that uses subject_writer, then html_converter,
    # then send_html_listing_email, in that order
)

listing_ops_manager = Agent(
    name="Listing Operations Manager",
    instructions=ops_instructions,
    tools=ops_tools,
    model="gpt-4o-mini",
    handoff_description="Format a listing email as HTML and send it to the prospect",
)

### Now we have 3 listing-agent tools + 1 handoff to the Listing Operations Manager

In [ ]:
# --- TODO: rewire the Listings Manager to hand off instead of sending directly ---

description = "Write a real-estate listing description"

tool1 = listing_agent1.as_tool(tool_name="listing_agent1", tool_description=description)
tool2 = listing_agent2.as_tool(tool_name="listing_agent2", tool_description=description)
tool3 = listing_agent3.as_tool(tool_name="listing_agent3", tool_description=description)

tools = [tool1, tool2, tool3]
handoffs = [___]  # TODO: pass listing_ops_manager

listings_manager_instructions = """
___
"""  # TODO: same idea as Part 3a, but step 3 is now a HANDOFF to 'Listing Operations Manager'
# instead of sending directly. Must hand off exactly ONE listing -- never more than one.

listings_manager = Agent(
    name="Listings Manager",
    instructions=listings_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
)

message = ___  # TODO: e.g. "Send a listing email for 12 Oak Lane from Skyline Realty"

with trace("Skyline Realty - Full Pipeline"):
    result = await Runner.run(listings_manager, message)

### Check the trace and your inbox

https://platform.openai.com/traces

---
# Solution

No peeking until you've tried it yourself!

In [ ]:
# === Setup ===
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

load_dotenv(override=True)

In [ ]:
# === Part 1a/1b/1c -- personas, agents, evaluator ===

instructions1 = (
    "You are a luxury real-estate copywriter at Skyline Realty. "
    "You write elegant, aspirational listing descriptions that emphasise prestige, craftsmanship, and exclusivity."
)

instructions2 = (
    "You are a family-focused real-estate copywriter at Skyline Realty. "
    "You write warm, practical listing descriptions that highlight schools, safety, space, and community."
)

instructions3 = (
    "You are an investor-focused real-estate copywriter at Skyline Realty. "
    "You write concise, numbers-driven listing descriptions that emphasise rental yield, growth potential, and ROI."
)

listing_agent1 = Agent(name="Luxury Copywriter", instructions=instructions1, model="gpt-4o-mini")
listing_agent2 = Agent(name="Family Copywriter", instructions=instructions2, model="gpt-4o-mini")
listing_agent3 = Agent(name="Investor Copywriter", instructions=instructions3, model="gpt-4o-mini")

buyer_picker = Agent(
    name="Prospective Buyer Evaluator",
    instructions=(
        "You evaluate listing descriptions as if you are a serious prospective home buyer. "
        "Pick the single description most likely to prompt a viewing request - one that feels "
        "credible, specific, and relevant. Reply with the selected description only, no explanation."
    ),
    model="gpt-4o-mini",
)

In [ ]:
# === Streamed single draft ===

async def run():
    result = Runner.run_streamed(
        listing_agent1,
        input="Write a listing description for a 3-bed house at 12 Oak Lane",
    )
    async for event in result.stream_events():
        if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)

task = asyncio.ensure_future(run())
await task

In [ ]:
# === Parallel drafts + evaluator ===

message = "Write a listing description for a 3-bed house at 12 Oak Lane"

with trace("Listing draft selection"):
    results = await asyncio.gather(
        Runner.run(listing_agent1, message),
        Runner.run(listing_agent2, message),
        Runner.run(listing_agent3, message),
    )
    outputs = [result.final_output for result in results]
    drafts = "Listing drafts:\n\n" + "\n\nDraft:\n\n".join(outputs)

    best = await Runner.run(buyer_picker, drafts)
    print(f"Best listing description:\n{best.final_output}")

In [ ]:
# === Part 2 -- tools ===

@function_tool
def send_listing_email(body: str) -> Dict[str, str]:
    """ Send a plain-text listing outreach email to a prospect """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("pashaajaz@gmail.com")  # Change to your verified sender
    to_email = To("pashaajaz@gmail.com")        # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "New Listing - Skyline Realty", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

description = "Write a real-estate listing description"

tool1 = listing_agent1.as_tool(tool_name="listing_agent1", tool_description=description)
tool2 = listing_agent2.as_tool(tool_name="listing_agent2", tool_description=description)
tool3 = listing_agent3.as_tool(tool_name="listing_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_listing_email]
tools

In [ ]:
# === Part 3a -- orchestrator, tools only ===

listings_manager_instructions = """
You are the Listings Manager at Skyline Realty.
Your goal is to send the single most effective listing email to a prospective buyer.

Follow these steps carefully:
1. Generate Drafts: use all three listing-agent tools to produce three different descriptions.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: choose the draft most likely to resonate with the prospect.
   You may use the tools again if you're not satisfied with the initial results.

3. Send: use the send_listing_email tool to send ONLY the winning draft.

Crucial Rules:
- You must use the listing-agent tools to draft descriptions - do not write them yourself.
- You must send exactly ONE email - never more than one.
"""

listings_manager = Agent(
    name="Listings Manager",
    instructions=listings_manager_instructions,
    tools=tools,
    model="gpt-4o-mini",
)

message = "Send a listing email for 12 Oak Lane, a 3-bed family home"

with trace("Skyline Realty listing - Listings Manager"):
    result = await Runner.run(listings_manager, message)

In [ ]:
# === Part 3b -- sub-agents, Listing Operations Manager, handoff ===

subject_instructions = (
    "You write compelling email subject lines for real-estate listing outreach. "
    "The subject should feel inviting and specific - not salesy. "
    "Return the subject line only, no explanation."
)

html_instructions = (
    "You convert a plain-text listing email body (which may contain markdown) into a clean HTML email. "
    "Use a simple, elegant layout befitting a premium real-estate brand. No flashy colours."
)

subject_writer = Agent(name="Subject Line Writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject line for a listing email")

html_converter = Agent(name="HTML Email Formatter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Convert a plain-text email body to HTML")

@function_tool
def send_html_listing_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send a formatted HTML listing email to a prospect """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("pashaajaz@gmail.com")  # Change to your verified sender
    to_email = To("pashaajaz@gmail.com")        # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

ops_tools = [subject_tool, html_tool, send_html_listing_email]

ops_instructions = (
    "You are an Email Operations Manager. You receive a draft listing email body. "
    "First, use the subject_writer tool to generate a subject line. "
    "Then, use the html_converter tool to convert the body to HTML. "
    "Finally, use the send_html_listing_email tool to send the email with the subject and HTML body."
)

listing_ops_manager = Agent(
    name="Listing Operations Manager",
    instructions=ops_instructions,
    tools=ops_tools,
    model="gpt-4o-mini",
    handoff_description="Format a listing email as HTML and send it to the prospect",
)

In [ ]:
# === Full pipeline: Listings Manager hands off to Listing Operations Manager ===

description = "Write a real-estate listing description"

tool1 = listing_agent1.as_tool(tool_name="listing_agent1", tool_description=description)
tool2 = listing_agent2.as_tool(tool_name="listing_agent2", tool_description=description)
tool3 = listing_agent3.as_tool(tool_name="listing_agent3", tool_description=description)

tools = [tool1, tool2, tool3]
handoffs = [listing_ops_manager]

listings_manager_instructions = """
You are the Listings Manager at Skyline Realty.
Your goal is to send the single most effective listing email to a prospective buyer.

Follow these steps carefully:
1. Generate Drafts: use all three listing-agent tools to produce three different descriptions.
   Do not proceed until all three drafts are ready.

2. Evaluate and Select: choose the draft most likely to resonate with the prospect.
   You may use the tools again if you're not satisfied with the initial results.

3. Handoff: pass ONLY the winning draft to the 'Listing Operations Manager' agent for formatting and sending.

Crucial Rules:
- You must use the listing-agent tools to draft descriptions - do not write them yourself.
- You must hand off exactly ONE listing - never more than one.
"""

listings_manager = Agent(
    name="Listings Manager",
    instructions=listings_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini",
)

message = "Send a listing email for 12 Oak Lane from Skyline Realty"

with trace("Skyline Realty - Full Pipeline"):
    result = await Runner.run(listings_manager, message)